# Module 03 — Assignment: Convolutions

Module 02's assignment had you build the toolkit; now you build the eye. You
implement the four load-bearing pieces of the convolution machinery — the
**conv forward** accumulation, the **dK/db backward**, **max-pool's gradient
routing**, and **He init for kernels** — and the check cells grade you against
finite-difference gradients and the canonical `python/lenet.py` + `nanograd`
answer key. The finale drops your pieces into a GIVEN harness that must train
the bars toy to 100%.

Rules: don't peek at `lib/python/nanograd` or `python/lenet.py` until you've
tried. The check cells are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '03-lenet')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import lenet as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng
print('setup ok — nanograd and check utils imported')

## Part 1 — the convolution, forward

One feature-map cell is one dot product: the kernel laid over the patch at
$(u, v)$, everything multiplied and summed, plus the filter's bias:

$$Y[i,f,u,v] = b_f + \sum_{c,p,q} K[f,c,p,q]\, X[i,c,u{+}p,v{+}q]$$

The outer loops are GIVEN; write the inner accumulation. (This is
cross-correlation — no kernel flip — like every modern framework.)

In [ ]:
def my_conv2d_forward(X, K, b):
    n, C, H, W = X.shape
    F, _, KH, KW = K.shape
    OH, OW = H - KH + 1, W - KW + 1        # GIVEN: valid conv, stride 1
    Y = np.empty((n, F, OH, OW))
    for i in range(n):                      # GIVEN: the canonical loop nest
        for f in range(F):
            for u in range(OH):
                for v in range(OW):
                    acc = None
    ###########################################################################
    # TODO: start acc at b[f], then add K[f, c, p, q] * X[i, c, u + p, v + q]
    # over all c, p, q (three nested loops). ~5 lines.
    ###########################################################################
    # acc = ...
    # for c in range(C):
    #     ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
                    Y[i, f, u, v] = acc
    return Y

In [ ]:
# Check: your forward vs the canonical definition AND the im2col fast path.
rng = np.random.default_rng(0)
X = rng.standard_normal((2, 2, 6, 6)); K = rng.standard_normal((3, 2, 3, 3))
b = rng.standard_normal(3)
Y_you = my_conv2d_forward(X, K, b)
Y_ref = canon.naive_conv2d_forward(X, K, b)
conv = ng.Conv2D(2, 3, 3, 3); conv.K[:] = K; conv.b[:] = b
print(f'vs naive definition: max|diff| = {np.abs(Y_you - Y_ref).max():.2e}')
print(f'vs im2col Conv2D:    max|diff| = {np.abs(Y_you - conv.forward(X)).max():.2e}')
assert np.abs(Y_you - Y_ref).max() < 1e-12
assert np.abs(Y_you - conv.forward(X)).max() < 1e-12
compare_to_canonical([float(Y_you.sum()), float(Y_you[1, 2, 0, 3])],
                     [float(Y_ref.sum()), float(Y_ref[1, 2, 0, 3])],
                     labels=['sum', 'Y[1,2,0,3]'])

### Inline Question 1
A dense layer mapping the 28×28 input to the same 6×24×24 output would need
$784 \times 3{,}456 \approx 2.7$ million weights. Our conv1 does it with
**156**. Where did the other 2.7 million go — which two assumptions about
images let us delete them? *Your answer:* ____

## Part 2 — the backward pass for the kernel

Each output cell's gradient flows back to every weight that helped produce it.
For the kernel that means: correlate the **input** with the **upstream
gradient** (and sum the upstream gradient for the bias):

$$\frac{\partial L}{\partial K[f,c,p,q]} = \sum_{i,u,v} dY[i,f,u,v]\,X[i,c,u{+}p,v{+}q]
\qquad
\frac{\partial L}{\partial b_f} = \sum_{i,u,v} dY[i,f,u,v]$$

(The toy's conv layer touches the data directly, so you don't need dX here —
but see Inline Question 2.)

In [ ]:
def my_conv2d_backward_params(X, K, dY):
    n, C, H, W = X.shape
    F, _, KH, KW = K.shape
    OH, OW = H - KH + 1, W - KW + 1
    dK = np.zeros_like(K)
    db = np.zeros(F)
    for i in range(n):                      # GIVEN: same nest as forward
        for f in range(F):
            for u in range(OH):
                for v in range(OW):
                    g = dY[i, f, u, v]
    ###########################################################################
    # TODO: accumulate g into db[f], then add g * X[i, c, u + p, v + q] into
    # dK[f, c, p, q] over all c, p, q. ~5 lines.
    ###########################################################################
    # db[f] += ...
    # for c in range(C):
    #     ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return dK, db

In [ ]:
# Check: finite differences, then the canonical backward.
dY = np.random.default_rng(2).standard_normal((2, 3, 4, 4))
dK_you, db_you = my_conv2d_backward_params(X, K, dY)
dK_num = eval_numerical_gradient_array(lambda _: my_conv2d_forward(X, K, b), K, dY)
db_num = eval_numerical_gradient_array(lambda _: my_conv2d_forward(X, K, b), b, dY)
print(f'dK vs finite differences: rel_error = {rel_error(dK_you, dK_num):.2e}')
print(f'db vs finite differences: rel_error = {rel_error(db_you, db_num):.2e}')
assert rel_error(dK_you, dK_num) < 1e-6 and rel_error(db_you, db_num) < 1e-6
_, dK_ref, db_ref = canon.naive_conv2d_backward(X, K, dY)
compare_to_canonical([float(dK_you.sum()), float(db_you.sum())],
                     [float(dK_ref.sum()), float(db_ref.sum())],
                     labels=['dK_sum', 'db_sum'])

### Inline Question 2
The input gradient works out to
$\partial L/\partial X[i,c,h,w] = \sum_{f,p,q} dY[i,f,h{-}p,w{-}q]\,K[f,c,p,q]$
— a *full* correlation of $dY$ with the kernel **flipped** in both axes, i.e. a
true convolution. We never flipped anything in forward. Where does the flip
come from? (Hint: which output positions did pixel $(h, w)$ contribute to, and
through which kernel tap?) *Your answer:* ____

## Part 3 — max-pool's backward: winner takes the gradient

Forward (GIVEN) keeps each 2×2 window's max and remembers *where* it was.
Backward is pure routing: each output gradient goes to the one input cell that
won the max; every other cell gets zero — the same mask idea as ReLU.

In [ ]:
class MyMaxPool2x2:
    def forward(self, X):
        # GIVEN: expose each 2x2 window as one axis, take max, cache argmax.
        n, C, H, W = X.shape
        self.x_shape = X.shape
        win = (X.reshape(n, C, H // 2, 2, W // 2, 2)
                .transpose(0, 1, 2, 4, 3, 5).reshape(n, C, H // 2, W // 2, 4))
        self.argmax = win.argmax(axis=-1)
        return np.take_along_axis(win, self.argmax[..., None], axis=-1)[..., 0]

    def backward(self, dY):
        n, C, H, W = self.x_shape
        dwin = np.zeros((n, C, H // 2, W // 2, 4))
    ###########################################################################
    # TODO: scatter each dY into dwin at the cached argmax position (hint:
    # np.put_along_axis, mirroring forward's np.take_along_axis), then undo the
    # window packing: reshape to (n, C, H//2, W//2, 2, 2), transpose the same
    # axes as forward, reshape to (n, C, H, W), and return it. ~2-4 lines.
    ###########################################################################
    # ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################

In [ ]:
# Check: routing (one winner per window), finite differences, canonical layer.
Xp = np.random.default_rng(3).standard_normal((2, 3, 6, 6))
dYp = np.random.default_rng(4).standard_normal((2, 3, 3, 3))
pool = MyMaxPool2x2(); pool.forward(Xp)
dX_you = pool.backward(dYp)
per_win = (dX_you.reshape(2, 3, 3, 2, 3, 2) != 0).sum(axis=(3, 5))
print('nonzero grads per 2x2 window (should all be 1):', np.unique(per_win))
num = eval_numerical_gradient_array(lambda _: MyMaxPool2x2().forward(Xp), Xp, dYp)
print(f'dX vs finite differences: rel_error = {rel_error(dX_you, num):.2e}')
assert rel_error(dX_you, num) < 1e-6
ref = ng.MaxPool2D(2); ref.forward(Xp)
compare_to_canonical([float(dX_you.sum())], [float(ref.backward(dYp).sum())],
                     labels=['dX_sum'])

### Inline Question 3
Shift a digit one pixel to the right. Conv feature maps shift with it — but
much of the *pooled* map doesn't change at all. Explain in one sentence why
pooling buys translation tolerance, and name the price paid. *Your answer:* ____

## Part 4 — He initialization for a kernel

Same rule as Module 02, new fan-in: a conv output unit reads $C \cdot KH
\cdot KW$ inputs (every channel of one patch), so
$\mathrm{std} = \sqrt{2 / (C \cdot KH \cdot KW)}$. Fill the kernel **in
place**, drawing in flat row-major (F,C,KH,KW) order from `rng.normal()` so it
matches the library (and the C mirror) draw for draw.

In [ ]:
def he_conv_init(K, rng):
    F, C, KH, KW = K.shape
    flat = K.reshape(-1)                    # GIVEN: a view — writes hit K
    ###########################################################################
    # TODO: set std = sqrt(2 / (C*KH*KW)), then fill flat[j] = std * rng.normal()
    # for every j in order. ~3 lines.
    ###########################################################################
    # std = ...
    # ...
    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return K

In [ ]:
# Check: your fill must match nanograd.he_normal_conv2d from the same seed.
Ka = np.zeros((4, 1, 3, 3)); he_conv_init(Ka, Rng(42))
Kb = np.zeros((4, 1, 3, 3)); ng.he_normal_conv2d(Kb, Rng(42))
print(f'std(you)={Ka.std():.4f}   std(nanograd)={Kb.std():.4f}   target={math.sqrt(2/9):.4f}')
compare_to_canonical([Ka[0, 0, 0, 0], Ka[2, 0, 1, 2], float(Ka.sum())],
                     [Kb[0, 0, 0, 0], Kb[2, 0, 1, 2], float(Kb.sum())],
                     labels=['K[0,0,0,0]', 'K[2,0,1,2]', 'sum'])

### Inline Question 4
Work out conv2's **receptive field**: one cell of conv2's output sees a 5×5
patch of the pooled map; each pooled cell covers a 2×2 patch of conv1's output;
each conv1 cell sees a 5×5 patch of the input. How many input pixels feed one
conv2 output cell (give the side length)? *Your answer:* ____

## Verify: your pieces compose to train the CNN

Each part above graded one piece against the canonical library. This last cell
is the composition test: your conv forward, conv backward, max-pool routing,
and He init are dropped into a GIVEN harness (the same tiny CNN as
`python/lenet.py`) and must drive the bars toy to **100% accuracy**. (We don't
compare final weights to the C/scalar answer key — the per-piece checks above
are the exact gates; this proves the whole gradient path works.)

In [ ]:
def train_bars_with(conv_fwd, conv_bwd, pool_cls, init_fn, steps=120):
    '''GIVEN harness: trains the toy CNN using YOUR pieces. Do not modify.'''
    Xb, yb = canon.make_bars()
    rng = Rng(canon.TOY_SEED)
    K = np.zeros((4, 1, 3, 3)); b = np.zeros(4)
    init_fn(K, rng)
    head = ng.Linear(36, 2); ng.he_normal(head.W, rng); head.b[:] = 0.0
    pool, lossf, opt = pool_cls(), ng.SoftmaxCrossEntropy(), ng.Adam(lr=0.02)
    for t in range(steps):
        Z = conv_fwd(Xb, K, b)
        A = np.maximum(Z, 0.0)                       # ReLU
        P = pool.forward(A)
        logits = head.forward(P.reshape(len(yb), -1))
        lossf.forward(logits, yb)
        dP = head.backward(lossf.backward()).reshape(P.shape)
        dA = pool.backward(dP)
        dZ = dA * (Z > 0.0)                          # ReLU mask
        dK, db = conv_bwd(Xb, K, dZ)
        opt.step([K, b, head.W, head.b], [dK, db, head.dW, head.db])
    logits = head.forward(pool.forward(np.maximum(conv_fwd(Xb, K, b), 0.0))
                          .reshape(len(yb), -1))
    loss = lossf.forward(logits, yb)
    return loss, float((logits.argmax(1) == yb).mean())

loss, acc = train_bars_with(my_conv2d_forward, my_conv2d_backward_params,
                            MyMaxPool2x2, he_conv_init)
print(f'trained bars with YOUR pieces:  acc = {acc:.3f}   loss = {loss:.6f}')
assert acc == 1.0, 'your pieces did not reach 100% on bars — check parts 1-4'
assert loss < 0.2, f'bars trained but loss {loss:.4g} is too high — check part 2'
print('OK — conv forward, conv backward, pooling, and init all pull together.')